In [1]:
import pandas as pd

### All the database

In [7]:
data = {}
for tp in range(4):
    data[tp+1] = pd.read_csv(f'datos/counted/d{tp+1}_counted.csv', index_col=0, lineterminator='\n')

/tmp/ipykernel_21914/2320036608.py:3: DtypeWarning: Columns (1,2,5,6,9) have mixed types. Specify dtype option on import or set low_memory=False.
  data[tp+1] = pd.read_csv(f'datos/counted/d{tp+1}_counted.csv', index_col=0, lineterminator='\n')
/tmp/ipykernel_21914/2320036608.py:3: DtypeWarning: Columns (1,2,3,5,6,7,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,26) have mixed types. Specify dtype option on import or set low_memory=False.
  data[tp+1] = pd.read_csv(f'datos/counted/d{tp+1}_counted.csv', index_col=0, lineterminator='\n')


In [14]:
for tp in range(4):
    data[tp+1]['tp']  = tp+1

In [15]:
data[1].columns

Index(['avisoid', 'empresaid', 'avisofechapublicacion', 'avisovacante',
       'mostrarsueldo', 'avisoexperiencia', 'expiracion', 'dias',
       'avisorepublicacion', 'avisocargo', 'areanombre',
       'actividadempresanombre', 'avisocuerpo', 'disponibilidadnombre',
       'avisoduracioncont', 'avisolugartrabajo', 'gradoescolarnombre',
       'situacionestudios', 'avisorequisitos', 'carreras', 'estado',
       'endpagado_o_gratuito', 'nivelnombre', 'carreras_array',
       'sueldoestimado', '_merge', 'teletrabajo_', 'avcu_b1', 'avca_b1',
       'avre_b1', 'avcu_b2', 'avca_b2', 'avre_b2', 'avcu_b3', 'avca_b3',
       'avre_b3', 'avcu_b4', 'avca_b4', 'avre_b4', 'tp'],
      dtype='object')

Data Frame to sum duplicates

In [16]:
datau = pd.concat([data[1], data[2], data[3], data[4]], axis=0, ignore_index=True)

### Window Size

In [28]:
datau['avisofechapublicacion'] = pd.to_datetime(datau['avisofechapublicacion'], errors='coerce')

In [29]:
# Calculate the start and end dates
start_date = datau['avisofechapublicacion'].min()
end_date = datau['avisofechapublicacion'].max()

# Create a date range with intervals
date_range_3m = pd.date_range(start=start_date, end=end_date, freq='3M') #3M stands for three months
date_range_2m = pd.date_range(start=start_date, end=end_date, freq='2M') #2M stands for two months
date_range_1m = pd.date_range(start=start_date, end=end_date, freq='1M') #1M stands for one month
date_range_6m = pd.date_range(start=start_date, end=end_date, freq='6M') #6M stands for six months

# Create a Pandas Series with the date range
dates1m = pd.Series(date_range_1m)
dates2m = pd.Series(date_range_2m)
dates3m = pd.Series(date_range_3m)
dates6m = pd.Series(date_range_6m)

/tmp/ipykernel_21914/2223016239.py:6: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  date_range_3m = pd.date_range(start=start_date, end=end_date, freq='3M') #3M stands for three months
/tmp/ipykernel_21914/2223016239.py:7: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  date_range_2m = pd.date_range(start=start_date, end=end_date, freq='2M') #2M stands for two months
/tmp/ipykernel_21914/2223016239.py:8: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  date_range_1m = pd.date_range(start=start_date, end=end_date, freq='1M') #1M stands for one month
/tmp/ipykernel_21914/2223016239.py:9: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  date_range_6m = pd.date_range(start=start_date, end=end_date, freq='6M') #6M stands for six months


______________________

avisocuerpo, empresaid, avisolugartrabajo

In [32]:
datau['duplicates_avisocuerpo1m'] = False
datau['duplicates_avisocuerpo2m'] = False
datau['duplicates_avisocuerpo3m'] = False
datau['duplicates_avisocuerpo6m'] = False

Each month (Window 1M)

In [33]:
for x in range(len(dates1m)-1):
    start_date = dates1m[x]
    final_date = dates1m[x+1]

    # Sub set with the date window
    newData = datau[(datau['avisofechapublicacion'] >= start_date) & (datau['avisofechapublicacion'] < final_date)]

    # Another Data frame with the duplicates for the newData subset
    duplicates = newData[newData.duplicated(subset=['empresaid', 'avisocuerpo', 'avisolugartrabajo'], keep='first')]  # Dont consider the first realiztion, mark everything else as duplicate

    # Check wich ad from Data dataFrame is in the duplicates dataFrame
    datau['duplicates_avisocuerpo1m'] = datau['duplicates_avisocuerpo1m'] + datau.index.isin(duplicates.index)

In [34]:
datau['duplicates_avisocuerpo1m'].value_counts()

duplicates_avisocuerpo1m
False    3525491
True      545926
Name: count, dtype: int64

Each two months (Window 2M)

In [35]:
for x in range(len(dates2m)-1):
    start_date = dates2m[x]
    final_date = dates2m[x+1]

    # Sub set with the date window
    newData = datau[(datau['avisofechapublicacion'] >= start_date) & (datau['avisofechapublicacion'] < final_date)]

    # Another Data frame with the duplicates for the newData subset
    duplicates = newData[newData.duplicated(subset=['empresaid', 'avisocuerpo', 'avisolugartrabajo'], keep='first')]  # Dont consider the first realiztion, mark everything else as duplicate

    # Check wich ad from Data dataFrame is in the duplicates dataFrame
    datau['duplicates_avisocuerpo2m'] = datau['duplicates_avisocuerpo2m'] + datau.index.isin(duplicates.index)

In [36]:
datau['duplicates_avisocuerpo2m'].value_counts()

duplicates_avisocuerpo2m
False    3464946
True      606471
Name: count, dtype: int64

Each quarter (Window 3M)

In [37]:
for x in range(len(dates3m)-1):
    start_date = dates3m[x]
    final_date = dates3m[x+1]

    # Sub set with the date window
    newData = datau[(datau['avisofechapublicacion'] >= start_date) & (datau['avisofechapublicacion'] < final_date)]

    # Another Data frame with the duplicates for the newData subset
    duplicates = newData[newData.duplicated(subset=['empresaid', 'avisocuerpo', 'avisolugartrabajo'], keep='first')]  # Dont consider the first realiztion, mark everything else as duplicate

    # Check wich ad from Data dataFrame is in the duplicates dataFrame
    datau['duplicates_avisocuerpo3m'] = datau['duplicates_avisocuerpo3m'] + datau.index.isin(duplicates.index)


In [38]:
datau['duplicates_avisocuerpo3m'].value_counts()

duplicates_avisocuerpo3m
False    3434908
True      636509
Name: count, dtype: int64

Each semester (Window 6M)

In [39]:
for x in range(len(dates6m)-1):
    start_date = dates6m[x]
    final_date = dates6m[x+1]

    # Sub set with the date window
    newData = datau[(datau['avisofechapublicacion'] >= start_date) & (datau['avisofechapublicacion'] < final_date)]

    # Another Data frame with the duplicates for the newData subset
    duplicates = newData[newData.duplicated(subset=['empresaid', 'avisocuerpo', 'avisolugartrabajo'], keep='first')]  # Dont consider the first realiztion, mark everything else as duplicate

    # Check wich ad from Data dataFrame is in the duplicates dataFrame
    datau['duplicates_avisocuerpo6m'] = datau['duplicates_avisocuerpo6m'] + datau.index.isin(duplicates.index)


In [40]:
datau['duplicates_avisocuerpo6m'].value_counts()

duplicates_avisocuerpo6m
False    3388510
True      682907
Name: count, dtype: int64

____________________________

avisocuerpo, empresaid, avisolugartrabajo, sueldoestimado

In [41]:
datau['duplicates_sueldoestimado1m'] = False
datau['duplicates_sueldoestimado2m'] = False
datau['duplicates_sueldoestimado3m'] = False
datau['duplicates_sueldoestimado6m'] = False

Each month (Window 1M)

In [42]:
for x in range(len(dates1m)-1):
    start_date = dates1m[x]
    final_date = dates1m[x+1]

    # Sub set with the date window
    newData = datau[(datau['avisofechapublicacion'] >= start_date) & (datau['avisofechapublicacion'] < final_date)]

    # Another Data frame with the duplicates for the newData subset
    duplicates = newData[newData.duplicated(subset=['empresaid', 'avisocuerpo', 'avisolugartrabajo', 'sueldoestimado'], keep='first')]  # Dont consider the first realiztion, mark everything else as duplicate

    # Check wich ad from Data dataFrame is in the duplicates dataFrame
    datau['duplicates_sueldoestimado1m'] = datau['duplicates_sueldoestimado1m'] + datau.index.isin(duplicates.index)


In [43]:
datau['duplicates_sueldoestimado1m'].value_counts()

duplicates_sueldoestimado1m
False    3541380
True      530037
Name: count, dtype: int64

Each two months (Window 2M)

In [44]:
for x in range(len(dates2m)-1):
    start_date = dates2m[x]
    final_date = dates2m[x+1]

    # Sub set with the date window
    newData = datau[(datau['avisofechapublicacion'] >= start_date) & (datau['avisofechapublicacion'] < final_date)]

    # Another Data frame with the duplicates for the newData subset
    duplicates = newData[newData.duplicated(subset=['empresaid', 'avisocuerpo', 'avisolugartrabajo', 'sueldoestimado'], keep='first')]  # Dont consider the first realiztion, mark everything else as duplicate

    # Check wich ad from Data dataFrame is in the duplicates dataFrame
    datau['duplicates_sueldoestimado2m'] = datau['duplicates_sueldoestimado2m'] + datau.index.isin(duplicates.index)


In [45]:
datau['duplicates_sueldoestimado2m'].value_counts()

duplicates_sueldoestimado2m
False    3484703
True      586714
Name: count, dtype: int64

Each quarter (Window 3M)

In [46]:
for x in range(len(dates3m)-1):
    start_date = dates3m[x]
    final_date = dates3m[x+1]

    # Sub set with the date window
    newData = datau[(datau['avisofechapublicacion'] >= start_date) & (datau['avisofechapublicacion'] < final_date)]

    # Another Data frame with the duplicates for the newData subset
    duplicates = newData[newData.duplicated(subset=['empresaid', 'avisocuerpo', 'avisolugartrabajo', 'sueldoestimado'], keep='first')]  # Dont consider the first realiztion, mark everything else as duplicate

    # Check wich ad from Data dataFrame is in the duplicates dataFrame
    datau['duplicates_sueldoestimado3m'] = datau['duplicates_sueldoestimado3m'] + datau.index.isin(duplicates.index)


In [48]:
datau['duplicates_sueldoestimado3m'].value_counts()

duplicates_sueldoestimado3m
False    3456700
True      614717
Name: count, dtype: int64

Each semester (Window 6M)

In [50]:
for x in range(len(dates6m)-1):
    start_date = dates6m[x]
    final_date = dates6m[x+1]

    # Sub set with the date window
    newData = datau[(datau['avisofechapublicacion'] >= start_date) & (datau['avisofechapublicacion'] < final_date)]

    # Another Data frame with the duplicates for the newData subset
    duplicates = newData[newData.duplicated(subset=['empresaid', 'avisocuerpo', 'avisolugartrabajo', 'sueldoestimado'], keep='first')]  # Dont consider the first realiztion, mark everything else as duplicate

    # Check wich ad from Data dataFrame is in the duplicates dataFrame
    datau['duplicates_sueldoestimado6m'] = datau['duplicates_sueldoestimado6m'] + datau.index.isin(duplicates.index)


In [51]:
datau['duplicates_sueldoestimado6m'].value_counts()

duplicates_sueldoestimado6m
False    3414494
True      656923
Name: count, dtype: int64

____________________

avisocargo, avisocuerpo, empresaid, avisolugartrabajo

In [52]:
datau['duplicates_avisocargo1m'] = False
datau['duplicates_avisocargo2m'] = False
datau['duplicates_avisocargo3m'] = False
datau['duplicates_avisocargo6m'] = False

Each month (Window 1M)

In [53]:
for x in range(len(dates1m)-1):
    start_date = dates1m[x] 
    final_date = dates1m[x+1]

    # Sub set with the date window
    newData = datau[(datau['avisofechapublicacion'] >= start_date) & (datau['avisofechapublicacion'] < final_date)]

    # Another Data frame with the duplicates for the newData subset
    duplicates = newData[newData.duplicated(subset=['empresaid', 'avisocargo', 'avisocuerpo', 'avisolugartrabajo'], keep='first')]  # Dont consider the first realiztion, mark everything else as duplicate

    # Check wich ad from Data dataFrame is in the duplicates dataFrame
    datau['duplicates_avisocargo1m'] = datau['duplicates_avisocargo1m'] + datau.index.isin(duplicates.index)

In [54]:
datau['duplicates_avisocargo1m'].value_counts()

duplicates_avisocargo1m
False    3677305
True      394112
Name: count, dtype: int64

Each two months (Window 2M)

In [55]:
for x in range(len(dates2m)-1):
    start_date = dates2m[x] 
    final_date = dates2m[x+1]

    # Sub set with the date window
    newData = datau[(datau['avisofechapublicacion'] >= start_date) & (datau['avisofechapublicacion'] < final_date)]

    # Another Data frame with the duplicates for the newData subset
    duplicates = newData[newData.duplicated(subset=['empresaid', 'avisocargo', 'avisocuerpo', 'avisolugartrabajo'], keep='first')]  # Dont consider the first realiztion, mark everything else as duplicate

    # Check wich ad from Data dataFrame is in the duplicates dataFrame
    datau['duplicates_avisocargo2m'] = datau['duplicates_avisocargo2m'] + datau.index.isin(duplicates.index)

In [56]:
datau['duplicates_avisocargo2m'].value_counts()

duplicates_avisocargo2m
False    3626947
True      444470
Name: count, dtype: int64

Each quarter (Window 3M)

In [57]:
for x in range(len(dates3m)-1):
    start_date = dates3m[x] 
    final_date = dates3m[x+1]

    # Sub set with the date window
    newData = datau[(datau['avisofechapublicacion'] >= start_date) & (datau['avisofechapublicacion'] < final_date)]

    # Another Data frame with the duplicates for the newData subset
    duplicates = newData[newData.duplicated(subset=['empresaid', 'avisocargo', 'avisocuerpo', 'avisolugartrabajo'], keep='first')]  # Dont consider the first realiztion, mark everything else as duplicate

    # Check wich ad from Data dataFrame is in the duplicates dataFrame
    datau['duplicates_avisocargo3m'] = datau['duplicates_avisocargo3m'] + datau.index.isin(duplicates.index)

In [58]:
datau['duplicates_avisocargo3m'].value_counts()

duplicates_avisocargo3m
False    3601238
True      470179
Name: count, dtype: int64

Each semester (Window 6M)

In [59]:
for x in range(len(dates6m)-1):
    start_date = dates6m[x] 
    final_date = dates6m[x+1]

    # Sub set with the date window
    newData = datau[(datau['avisofechapublicacion'] >= start_date) & (datau['avisofechapublicacion'] < final_date)]

    # Another Data frame with the duplicates for the newData subset
    duplicates = newData[newData.duplicated(subset=['empresaid', 'avisocargo', 'avisocuerpo', 'avisolugartrabajo'], keep='first')]  # Dont consider the first realiztion, mark everything else as duplicate

    # Check wich ad from Data dataFrame is in the duplicates dataFrame
    datau['duplicates_avisocargo6m'] = datau['duplicates_avisocargo6m'] + datau.index.isin(duplicates.index)

In [60]:
datau['duplicates_avisocargo6m'].value_counts()

duplicates_avisocargo6m
False    3561877
True      509540
Name: count, dtype: int64

Save the data

In [61]:
d1 = datau[datau['tp'] == 1] # Till september 2019
d2 = datau[datau['tp'] == 2] # October 2019 to March 2020
d3 = datau[datau['tp'] == 3] # March 2020 to 2021
d4 = datau[datau['tp'] == 4] # 2021 onwards

In [62]:
d1.to_csv(f'datos/counted/d1_allcounted.csv')
d2.to_csv(f'datos/counted/d2_allcounted.csv')
d3.to_csv(f'datos/counted/d3_allcounted.csv')
d4.to_csv(f'datos/counted/d4_allcounted.csv')